# Étape 6 - Contrôles des données Silver

Après écriture par `nettoyage_silver.ipynb` et `enrichissement_ventes.ipynb`, on vérifie pour chaque jeu Silver :

1. la **présence** des fichiers ;
2. leur **format** (Parquet) ;
3. leur **lisibilité** par Spark ;
4. leur **nombre de lignes** ;
5. leur **schéma** ;
6. leur **cohérence** avec les résultats attendus.

Chaque contrôle donne OK ou KO. Le notebook s'arrête si un contrôle échoue.

## 0. Préparation

In [1]:
import glob
import json
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

RACINE = os.getcwd()
while os.path.basename(RACINE) != "retail-intelligence-platform":
    RACINE = os.path.dirname(RACINE)
    if RACINE == "/":
        raise RuntimeError("Lancer le notebook depuis le dossier retail-intelligence-platform")

LOCAL = "file://"
DOSSIER_SILVER = os.path.join(RACINE, "data", "silver")
DOSSIER_SCHEMAS = os.path.join(RACINE, "docs", "data-dictionary", "schemas")

with open(os.path.join(RACINE, "outputs", "bilan_nettoyage.json")) as f:
    BILAN = json.load(f)

spark = (
    SparkSession.builder
    .appName("TP3-Controles-Silver")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "768m")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# Jeu Silver -> (table source, cle, colonnes obligatoires)
JEUX = {
    "sales": ("store_sales", ["ss_ticket_number", "ss_item_sk", "ss_ligne_ticket"],
              ["ss_sold_date_sk", "ss_item_sk", "ss_store_sk", "ss_quantity", "ss_net_paid", "ss_net_profit"]),
    "products": ("item", ["i_item_sk"], ["i_item_id", "i_category", "i_current_price"]),
    "stores": ("store", ["s_store_sk"], ["s_store_id", "s_store_name"]),
    "dates": ("date_dim", ["d_date_sk"], ["d_date", "d_year", "d_moy"]),
    "sales_enriched": (None, ["ticket_id", "produit_sk", "ligne_ticket"],
                       ["date_vente", "categorie", "nom_magasin", "montant_net_paye", "marge"]),
}
PROVENANCE = ["_source_systeme", "_source_fichier", "_traitement", "_date_traitement"]

resultats = []


def controle(jeu, nom, ok, detail=""):
    resultats.append({"jeu": jeu, "controle": nom, "resultat": "OK" if ok else "KO", "detail": str(detail)})
    print(f"[{'OK' if ok else 'KO'}] {jeu:<15} {nom:<34} {detail}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 16:44:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Présence et 2. format des fichiers

In [2]:
for jeu in JEUX:
    dossier = os.path.join(DOSSIER_SILVER, jeu)
    fichiers = glob.glob(os.path.join(dossier, "**", "*.parquet"), recursive=True)
    taille = sum(os.path.getsize(f) for f in fichiers)
    controle(jeu, "dossier present", os.path.isdir(dossier), dossier.replace(RACINE + "/", ""))
    controle(jeu, "ecriture terminee (_SUCCESS)", os.path.exists(os.path.join(dossier, "_SUCCESS")))
    controle(jeu, "fichiers .parquet presents", len(fichiers) > 0, f"{len(fichiers)} fichier(s), {taille / 1e6:.2f} Mo")

    # Un fichier Parquet commence et se termine par les 4 octets "PAR1"
    parquet_valide = True
    for chemin in fichiers:
        with open(chemin, "rb") as f:
            debut = f.read(4)
            f.seek(-4, os.SEEK_END)
            fin = f.read(4)
        parquet_valide &= debut == b"PAR1" and fin == b"PAR1"
    controle(jeu, "format Parquet (signature PAR1)", parquet_valide)

[OK] sales           dossier present                    data/silver/sales
[OK] sales           ecriture terminee (_SUCCESS)       
[OK] sales           fichiers .parquet presents         3 fichier(s), 4.41 Mo
[OK] sales           format Parquet (signature PAR1)    
[OK] products        dossier present                    data/silver/products
[OK] products        ecriture terminee (_SUCCESS)       
[OK] products        fichiers .parquet presents         4 fichier(s), 3.62 Mo
[OK] products        format Parquet (signature PAR1)    
[OK] stores          dossier present                    data/silver/stores
[OK] stores          ecriture terminee (_SUCCESS)       
[OK] stores          fichiers .parquet presents         3 fichier(s), 0.04 Mo
[OK] stores          format Parquet (signature PAR1)    
[OK] dates           dossier present                    data/silver/dates
[OK] dates           ecriture terminee (_SUCCESS)       
[OK] dates           fichiers .parquet presents         4 fichier(s

## 3. Lisibilité et 4. nombre de lignes

In [3]:
""" Nombre de lignes attendu :
- tables issues du nettoyage : lignes lues - lignes rejetees - doublons exacts supprimes (bilan de l'etape 4)
- sales_enriched : meme nombre de lignes que sales (jointures N:1 sans perte)
On verifie aussi que les lignes lues correspondent au fichier brut (comptage independant, sans Spark). """
donnees = {}
for jeu, (table, _, _) in JEUX.items():
    try:
        df = spark.read.parquet(LOCAL + os.path.join(DOSSIER_SILVER, jeu))
        nb = df.count()
        donnees[jeu] = df
        controle(jeu, "lisible par Spark", True)
    except Exception as e:
        controle(jeu, "lisible par Spark", False, e)
        continue

    if table:
        b = BILAN[table]
        attendu = b["lignes_lues"] - sum(b["rejets_par_motif"].values()) - b.get("doublons_exacts_supprimes", 0)
        rejets = spark.read.parquet(LOCAL + os.path.join(DOSSIER_SILVER, "rejects", jeu)).count()
        controle(jeu, "nombre de lignes = attendu", nb == attendu, f"{nb} lignes (attendu {attendu})")
        controle(jeu, "lignes Silver + rejets = lues", nb + rejets + b.get("doublons_exacts_supprimes", 0)
                 == b["lignes_lues"], f"{nb} + {rejets} rejet(s) = {b['lignes_lues']}")
        brut = spark.read.parquet(LOCAL + os.path.join(DOSSIER_SILVER, jeu)).select("_source_fichier").first()[0]
        with open(brut, "rb") as f:
            lignes_brut = sum(1 for _ in f)
        controle(jeu, "lignes lues = fichier brut", lignes_brut == b["lignes_lues"],
                 f"{lignes_brut} ligne(s) dans {os.path.basename(brut)}")
    else:
        nb_sales = donnees["sales"].count()
        controle(jeu, "nombre de lignes = sales", nb == nb_sales, f"{nb} lignes (attendu {nb_sales})")

[OK] sales           lisible par Spark                  


[OK] sales           nombre de lignes = attendu         66593 lignes (attendu 66593)
[OK] sales           lignes Silver + rejets = lues      66593 + 0 rejet(s) = 66593


[OK] sales           lignes lues = fichier brut         66593 ligne(s) dans store_sales_echantillon.dat


[OK] products        lisible par Spark                  


[OK] products        nombre de lignes = attendu         17820 lignes (attendu 17820)
[OK] products        lignes Silver + rejets = lues      17820 + 0 rejet(s) = 17820


[OK] products        lignes lues = fichier brut         17820 ligne(s) dans item.dat


... (4 outputs left out)

[OK] dates           nombre de lignes = attendu         109573 lignes (attendu 109573)
[OK] dates           lignes Silver + rejets = lues      109573 + 0 rejet(s) = 109573


[OK] dates           lignes lues = fichier brut         109573 ligne(s) dans date_dim.dat


[OK] sales_enriched  lisible par Spark                  


[OK] sales_enriched  nombre de lignes = sales           66593 lignes (attendu 66593)


## 5. Schéma

Le schéma relu est comparé au schéma de référence exporté lors de l'écriture (`docs/data-dictionary/schemas/`). On vérifie aussi les types attendus : montants en `decimal(7,2)`, dates en `date`.

In [4]:
TYPES_ATTENDUS = {
    "sales": {"ss_net_paid": "decimal(7,2)", "ss_net_profit": "decimal(7,2)", "ss_quantity": "int",
              "ss_item_sk": "bigint", "ss_ticket_number": "bigint"},
    "products": {"i_current_price": "decimal(7,2)", "i_rec_start_date": "date", "i_item_sk": "bigint"},
    "stores": {"s_zip": "string", "s_rec_start_date": "date", "s_closed_date": "date", "s_gmt_offset": "decimal(5,2)"},
    "dates": {"d_date": "date", "d_date_sk": "bigint"},
    "sales_enriched": {"date_vente": "date", "montant_net_paye": "decimal(7,2)", "marge": "decimal(7,2)",
                       "annee": "int"},
}

for jeu, df in donnees.items():
    with open(os.path.join(DOSSIER_SCHEMAS, f"silver_{jeu}.json")) as f:
        reference = json.load(f)
    relu = json.loads(df.schema.json())
    # L'ordre des colonnes peut changer avec le partitionnement : on compare nom -> type
    ref = {c["name"]: c["type"] for c in reference["fields"]}
    lu = {c["name"]: c["type"] for c in relu["fields"]}
    controle(jeu, "schema = schema de reference", ref == lu, f"{len(lu)} colonnes")

    types = dict(df.dtypes)
    ecarts = {c: (t, types.get(c)) for c, t in TYPES_ATTENDUS[jeu].items() if types.get(c) != t}
    controle(jeu, "types explicites attendus", not ecarts, ecarts or "")
    controle(jeu, "colonnes de provenance", all(c in types for c in PROVENANCE))

[OK] sales           schema = schema de reference       32 colonnes
[OK] sales           types explicites attendus          
[OK] sales           colonnes de provenance             
[OK] products        schema = schema de reference       28 colonnes
[OK] products        types explicites attendus          
[OK] products        colonnes de provenance             
[OK] stores          schema = schema de reference       35 colonnes
[OK] stores          types explicites attendus          
[OK] stores          colonnes de provenance             
[OK] dates           schema = schema de reference       14 colonnes
[OK] dates           types explicites attendus          
[OK] dates           colonnes de provenance             
[OK] sales_enriched  schema = schema de reference       42 colonnes
[OK] sales_enriched  types explicites attendus          
[OK] sales_enriched  colonnes de provenance             


In [5]:
donnees["sales"].printSchema()

root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = true)
 |-- ss_coupon_amt: decimal(7,2) (nullable = true)
 |-- ss_net_paid: decimal(7,2) (nullable = 

## 6. Cohérence avec les résultats attendus

In [6]:
def une_colonne(colonnes, test):
    cond = F.lit(False)
    for c in colonnes:
        cond = cond | test(F.col(c))
    return cond


for jeu, (_, cle, obligatoires) in JEUX.items():
    df = donnees[jeu]
    nb = df.count()
    doublons = nb - df.dropDuplicates(cle).count()
    controle(jeu, "cle unique", doublons == 0, f"{cle} : {doublons} doublon(s)")
    vides = df.filter(une_colonne(cle + obligatoires, lambda c: c.isNull())).count()
    controle(jeu, "colonnes obligatoires remplies", vides == 0, f"{vides} ligne(s) incomplete(s)")
    sans_provenance = df.filter(une_colonne(PROVENANCE, lambda c: c.isNull())).count()
    controle(jeu, "provenance renseignee", sans_provenance == 0)

[OK] sales           cle unique                         ['ss_ticket_number', 'ss_item_sk', 'ss_ligne_ticket'] : 0 doublon(s)


[OK] sales           colonnes obligatoires remplies     0 ligne(s) incomplete(s)


[OK] sales           provenance renseignee              


[OK] products        cle unique                         ['i_item_sk'] : 0 doublon(s)


[OK] products        colonnes obligatoires remplies     0 ligne(s) incomplete(s)


[OK] products        provenance renseignee              


... (5 outputs left out)

[OK] dates           provenance renseignee              


[OK] sales_enriched  cle unique                         ['ticket_id', 'produit_sk', 'ligne_ticket'] : 0 doublon(s)


[OK] sales_enriched  colonnes obligatoires remplies     0 ligne(s) incomplete(s)


[OK] sales_enriched  provenance renseignee              


In [7]:
ventes, enrichies = donnees["sales"], donnees["sales_enriched"]
produits, magasins, dates = donnees["products"], donnees["stores"], donnees["dates"]

# Le CA ne doit pas changer entre sales et sales_enriched
ca_sales = ventes.agg(F.sum("ss_net_paid")).first()[0]
ca_enrichi = enrichies.agg(F.sum("montant_net_paye")).first()[0]
controle("sales_enriched", "CA identique a sales", ca_sales == ca_enrichi, f"{ca_enrichi} / {ca_sales}")

# Integrite referentielle : chaque vente retrouve son produit, son magasin et sa date
for nom, ref, cle_v, cle_r in [("produit", produits, "ss_item_sk", "i_item_sk"),
                               ("magasin", magasins, "ss_store_sk", "s_store_sk"),
                               ("date", dates, "ss_sold_date_sk", "d_date_sk")]:
    orphelins = ventes.join(ref, ventes[cle_v] == ref[cle_r], "left_anti").count()
    controle("sales", f"chaque vente a son {nom}", orphelins == 0, f"{orphelins} orpheline(s)")

# Regles de l'etape 4 bien appliquees
controle("sales", "quantites > 0", ventes.filter("ss_quantity <= 0").count() == 0)
controle("sales", "montant = quantite x prix",
         ventes.filter(F.abs(F.col("ss_ext_sales_price") - F.col("ss_quantity") * F.col("ss_sales_price")) > 0.01)
         .count() == 0)
controle("stores", "dates de fermeture valides",
         magasins.filter(F.col("s_closed_date") < F.col("s_rec_start_date")).count() == 0)
controle("stores", "codes postaux a 5 caracteres", magasins.filter(F.length("s_zip") != 5).count() == 0)

periode = enrichies.agg(F.min("date_vente"), F.max("date_vente")).first()
controle("sales_enriched", "periode 2001-01-01 -> 2005-12-13",
         str(periode[0]) == "2001-01-01" and str(periode[1]) == "2005-12-13", f"{periode[0]} -> {periode[1]}")

[OK] sales_enriched  CA identique a sales               115752176.68 / 115752176.68


[OK] sales           chaque vente a son produit         0 orpheline(s)


[OK] sales           chaque vente a son magasin         0 orpheline(s)


[OK] sales           chaque vente a son date            0 orpheline(s)


[OK] sales           quantites > 0                      


[OK] sales           montant = quantite x prix          
[OK] stores          dates de fermeture valides         


[OK] stores          codes postaux a 5 caracteres       


[OK] sales_enriched  periode 2001-01-01 -> 2005-12-13   2001-01-01 -> 2005-12-13


## Synthèse

In [8]:
ko = [r for r in resultats if r["resultat"] == "KO"]
print(f"{len(resultats)} controle(s), {len(resultats) - len(ko)} OK, {len(ko)} KO")
with open(os.path.join(RACINE, "outputs", "controles_silver.json"), "w") as f:
    json.dump(resultats, f, indent=1, ensure_ascii=False)
assert not ko, ko

77 controle(s), 77 OK, 0 KO


In [9]:
spark.stop()